# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We will print the list of record sets, their fields, and the corresponding `@id`s.

In [ ]:
# List all record sets and their fields by @id

record_sets = list(dataset.record_sets)
print("Record sets in the dataset:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name}")
    print(f"  Fields (by @id):")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print("")

# If dataset has documentation, print that as well.
if getattr(metadata, 'distribution', None):
    print("Distributions (DataFileObjects):")
    for dist in metadata.distribution:
        print(f"- {dist['@id']}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract the first data table (the main protein record set).

**Note:** You can adapt this to any other record set by changing its `@id`.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, print all columns for the first record set
main_rs_id = record_set_ids[0]
print(f"Columns in record set {main_rs_id}:")
print(dataframes[main_rs_id].columns.tolist())

dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering, normalization, outlier removal, and grouping.

*Example*: Let's pick a numeric field such as "CoveragePercent" or "MW" (molecular weight) for filtering and normalization. We reference by the field `@id`.

In [ ]:
# Identify a numeric field from the fields in the chosen record set
chosen_record_set_id = main_rs_id

# Optionally, print field @ids and names for user reference
for rs in dataset.record_sets:
    if rs.id == chosen_record_set_id:
        print(f"Fields in record set '{chosen_record_set_id}':")
        for field in rs.fields:
            print(f"- {field.id}: {field.name}, type: {field.data_type}")

# For demonstration, let's try to use 'MW' (Molecular Weight) if it exists
numeric_field_id = None
df = dataframes[chosen_record_set_id]
for col in df.columns:
    if 'MW' in col or 'molecular_weight' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fall back to the first numeric-looking field
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

print(f"Using numeric field: {numeric_field_id}")

# Remove outliers (above 99th percentile)
threshold = df[numeric_field_id].quantile(0.99)
filtered_df = df[df[numeric_field_id] < threshold]
print(f"Filtered records with {numeric_field_id} < 99th percentile ({threshold}):")
print(filtered_df.head())

# Normalization (Z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by field, e.g., 'Description' or similar string/categorical field
possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
group_field = possible_group_fields[0] if possible_group_fields else None

if group_field:
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we will plot the distribution of the normalized numeric field and a bar plot of group means for the chosen group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the normalized numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=30, kde=True)
plt.title(f"Distribution of Normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id} (Normalized)")
plt.ylabel("Frequency")
plt.show()

# Bar plot of group means if group_field exists
if group_field:
    group_means = grouped_df.sort_values(by=numeric_field_id, ascending=False).head(10)
    plt.figure(figsize=(10,5))
    sns.barplot(data=group_means, x=group_field, y=numeric_field_id)
    plt.title(f"Top 10 {group_field} by mean {numeric_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load and explore the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library, referencing all fields and record sets by their Croissant `@id`s.

**Key learnings:**
- `mlcroissant` enables simple, robust exploration of FAIR datasets defined by Croissant schemas.
- Filtering, normalization, and grouping of data can be done seamlessly using standard pandas workflows after extraction.
- Metadata, columns, and fields should always be referenced by their `@id` to ensure schema consistency.

You can adapt this template to other datasets with different schemas by following the same fundamental steps and substituting the `@id` values for your data of interest.